# Outlet topics

Associated chart: the race/politics and LGBTQ outlet comparisons, and the who-gets-covered dot plot

Output files:
`outlet_topics.csv` and `outlet_gap.json` in `src/data/`.

Input files: every per-outlet CSV in `data/`, `src/data/headline_network.json`, `data/topic_terms.csv`.


## Setup

In [1]:
import csv, json, re
from collections import Counter, defaultdict

import pandas as pd

# the same vocabulary as the timeline: data/topic_terms.csv (see notebooks/topics.py)
from topics import patterns
PATS = patterns()
PATTERNS = {
    # the race panels: race terms only, BLM among them
    "polrace": re.compile(PATS["race / racism"], re.I),
    "lgbtq": re.compile(PATS["LGBTQ or trans"], re.I),
    # the contact words (the Sheet's "fouls" topic), for the same panels
    "fouls": re.compile(PATS["fouls"], re.I),
    "protect": re.compile(PATS["protect"], re.I),
}

csv.field_size_limit(10 ** 9)

131072

## Topic rates by outlet and year




In [2]:
# output names are what the chart components key on
import sys as _sys; _sys.path.insert(0, ".")
from outlets import FILES as OUTLET_FILES, SPORTS, NEWS, COMPARISON, OUTLETS as OUTLET_TABLE
CORPORA_FILES = {
    **OUTLET_FILES,                       # Fox News, the sports outlets, the news arms
    "Yahoo Sports (syndicated pool)": "yahoo_sports.csv",
    "Fox News (NBA)": "fox_nba.csv",
}
# every outlet has bodies now (The Athletic's came from harvest_athletic_bodies.py);
# an outlet listed here would stay headline-only
HEADLINE_ONLY = set()

corpora = {}
for name, filename in CORPORA_FILES.items():
    with open(f"../data/{filename}") as fh:
        corpora[name] = [((r["date"] or "")[:10], r["headline"] or "", r["body"] or "")
                         for r in csv.DictReader(fh) if r.get("date")]
{name: len(rows) for name, rows in corpora.items()}

{'Fox News': 4352,
 'ESPN': 4456,
 'CBS': 6210,
 'USA Today': 5777,
 'New York Times': 4228,
 'AP': 5123,
 'Washington Post': 2457,
 'Yahoo Sports (syndicated pool)': 29003,
 'Fox News (NBA)': 10739}

In [3]:
def tally(rows, pattern):
    """For each year, count how many articles the outlet published and how
    many of them mention the topic."""
    years = defaultdict(lambda: Counter())
    for date, text in rows:
        year = date[:4]
        if not year.isdigit() or year < "2019":   # the chart starts at 2019
            continue
        years[year]["n"] += 1
        if pattern.search(text):
            years[year]["hit"] += 1
    return dict(sorted(years.items()))

records = []
for topic, pattern in PATTERNS.items():
    for method in ("body", "headline"):
        for name, rows in corpora.items():
            if method == "body" and name in HEADLINE_ONLY:
                continue
            # full text counts only the articles that have it, so a CBS or USA
            # Today row we only got the headline for does not dilute the share
            texts = ((d, h) for d, h, b in rows) if method == "headline" else \
                    ((d, h + " " + b) for d, h, b in rows if len(b.split()) > 80)
            for year, v in tally(texts, pattern).items():
                records.append((topic, method, name, year, v["n"], v["hit"]))

#### Overwrites `src/data/outlet_topics.csv`

In [4]:
with open("../src/data/outlet_topics.csv", "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["topic", "method", "outlet", "year", "articles", "hits"])
    writer.writerows(records)
print(f"{len(records)} rows")

576 rows


In [5]:
topics = pd.read_csv("../src/data/outlet_topics.csv")
rates = topics.query("topic == 'all' and method == 'body' and articles >= 10")
rates.assign(pct=(100 * rates.hits / rates.articles).round(1)) \
     .pivot_table(index="outlet", columns="year", values="pct")

year
outlet


## The outlet gap



In [6]:
MIN_FOX_HEADLINES = 3
EXCLUDE = {"Layshia Clarendon"}   # in the network but not this chart
# Yahoo is out of the piece: mostly syndicated wire copy, and its stale end
# date was clipping the shared window for everyone
GROUPS = {name: members for name, members in (
    ("sports outlets", SPORTS), ("news outlets", NEWS), ("all outlets", COMPARISON)) if members}
# every outlet the picker can set Fox against; the shared window is the span they
# all cover, so Just Women's Sports (from 2020) pulls the start to 2020-01-30
GAP_FILES = {label: OUTLET_FILES[label] for label in COMPARISON}
# the shared window is anchored on the original five sports outlets: the news
# arms (and Fox Sports) run shorter or thinner and would clip it for everyone
SPORTS_BASELINE = set(SPORTS) or set(COMPARISON)   # the sports press when there is one, else everyone

NOT_HER = {
    "Sue Bird": r"larry bird|bird-magic|bird and magic",
    "A'ja Wilson": r"russell wilson|signs with wilson|wilson (?:basketball|ball|brand)",
    "Alyssa Thomas": r"lia thomas|isiah thomas",
    "Caitlin Clark": r"alysha clark",
}
NOT_HER_RX = {name: re.compile(rx, re.I) for name, rx in NOT_HER.items()}

DOT_PLOT_PLAYERS = [   # hand-picked: the chart's cast, in this order
    "Caitlin Clark", "Angel Reese", "Brittney Griner", "Sophie Cunningham", "Paige Bueckers",
    "Sue Bird", "DiJonai Carrington", "A'ja Wilson", "Sabrina Ionescu", "Aliyah Boston",
    "Breanna Stewart", "Alyssa Thomas", "Diana Taurasi", "Kelsey Plum", "Cameron Brink",
    "Candace Parker", "Natasha Cloud", "Napheesa Collier", "Chennedy Carter",
    "Nneka Ogwumike", "Kelsey Mitchell",
]
roster = json.load(open("../src/data/people.json"))
players = [n for n in DOT_PLOT_PLAYERS if n not in EXCLUDE]
name_rx = {n: re.compile(r"\b(?:" + roster[n]["rx"] + r")\b", re.I) for n in players}

def names_in(text):
    """Players named in this text, by surname, minus the look-alikes."""
    found = []
    for name, rx in name_rx.items():
        if not rx.search(text):
            continue
        guard = NOT_HER_RX.get(name)
        if guard and guard.search(text) and not re.search(
                re.escape(name.split()[0]), text, re.I):
            continue
        found.append(name)
    return found

players

['Caitlin Clark',
 'Angel Reese',
 'Brittney Griner',
 'Sophie Cunningham',
 'Paige Bueckers',
 'Sue Bird',
 'DiJonai Carrington',
 "A'ja Wilson",
 'Sabrina Ionescu',
 'Aliyah Boston',
 'Breanna Stewart',
 'Alyssa Thomas',
 'Diana Taurasi',
 'Kelsey Plum',
 'Cameron Brink',
 'Candace Parker',
 'Natasha Cloud',
 'Napheesa Collier',
 'Chennedy Carter',
 'Nneka Ogwumike',
 'Kelsey Mitchell']

In [7]:
# one (date, headline) list per outlet; Fox deduplicated by headline
rows_by_outlet = defaultdict(list)
seen = set()
for row in csv.DictReader(open("../data/fox_wnba.csv")):
    headline, date = (row["headline"] or "").strip(), (row["date"] or "")[:10]
    if headline and date and headline not in seen:
        seen.add(headline)
        rows_by_outlet["Fox News"].append((date, headline))
for outlet, filename in GAP_FILES.items():
    for row in csv.DictReader(open(f"../data/{filename}")):
        headline, date = (row["headline"] or "").strip(), (row["date"] or "")[:10]
        if headline and date:
            rows_by_outlet[outlet].append((date, headline))

# the window every outlet covers: latest start, earliest end
compared = ["Fox News"] + sorted(SPORTS_BASELINE)
window = (max(min(d for d, _ in rows_by_outlet[o]) for o in compared),
          min(max(d for d, _ in rows_by_outlet[o]) for o in compared))
print(f"comparing over {window[0]} .. {window[1]}")

totals, hits = Counter(), defaultdict(Counter)
for outlet, entries in rows_by_outlet.items():
    for date, headline in entries:
        if window[0] <= date <= window[1]:
            totals[outlet] += 1
            for name in names_in(headline):
                hits[outlet][name] += 1
dict(totals)

comparing over 2019-02-06 .. 2026-10-12


{'Fox News': 4330,
 'ESPN': 4412,
 'CBS': 6176,
 'USA Today': 5709,
 'New York Times': 4197,
 'AP': 5094,
 'Washington Post': 2455}

In [8]:
baseline_total = sum(totals[o] for o in SPORTS_BASELINE)
rows = []
for name in players:
    fox_share = hits["Fox News"][name] / totals["Fox News"]
    baseline_hits = sum(hits[o][name] for o in SPORTS_BASELINE)
    baseline_share = baseline_hits / baseline_total
    rows.append({
        "name": name,
        "fox": round(fox_share, 5),
        "foxN": hits["Fox News"][name],
        "sports": round(baseline_share, 5),
        "sportsN": baseline_hits,
        # each comparison outlet on its own, then the pooled groups
        "byOutlet": {
            **{o: {"share": round(hits[o][name] / totals[o], 5), "n": hits[o][name]}
               for o in GAP_FILES if totals[o]},
            **{label: {"share": round(sum(hits[o][name] for o in outlets)
                                      / max(1, sum(totals[o] for o in outlets)), 5),
                       "n": sum(hits[o][name] for o in outlets)}
               for label, outlets in GROUPS.items()
               if sum(totals[o] for o in outlets)},
        },
        # how many times more (or less) of Fox's coverage she is
        "ratio": round(fox_share / baseline_share, 2) if baseline_share else None,
        "news": {},   # the CNN / MS NOW harvests live outside this repo
    })
rows.sort(key=lambda r: -r["fox"])

In [9]:
# the same comparison over article text, where we have it
bodies = defaultdict(list)
for outlet, filename in {"Fox News": "fox_wnba.csv", **GAP_FILES}.items():
    for row in csv.DictReader(open(f"../data/{filename}")):
        text, date = (row["body"] or "").strip(), (row["date"] or "")[:10]
        if date and len(text.split()) > 80:
            bodies[outlet].append((date, text))

body_totals, body_hits = Counter(), defaultdict(Counter)
for outlet, entries in bodies.items():
    for date, text in entries:
        if window[0] <= date <= window[1]:
            body_totals[outlet] += 1
            for name in names_in(text):
                body_hits[outlet][name] += 1

for row in rows:
    name = row["name"]
    by_outlet = {o: {"share": round(body_hits[o][name] / body_totals[o], 5), "n": body_hits[o][name]}
                 for o in GAP_FILES if body_totals[o]}
    for label, outlets in GROUPS.items():
        denominator = sum(body_totals[o] for o in outlets)
        if denominator:
            by_outlet[label] = {
                "share": round(sum(body_hits[o][name] for o in outlets) / denominator, 5),
                "n": sum(body_hits[o][name] for o in outlets)}
    if body_totals["Fox News"] and by_outlet:
        row["body"] = {
            "fox": round(body_hits["Fox News"][name] / body_totals["Fox News"], 5),
            "foxN": body_hits["Fox News"][name],
            "byOutlet": by_outlet,
        }
dict(body_totals)

{'Fox News': 4329,
 'ESPN': 4392,
 'CBS': 6038,
 'USA Today': 4899,
 'New York Times': 3987,
 'AP': 5074,
 'Washington Post': 2408}

#### Overwrites `src/data/outlet_gap.json`

In [10]:
with open("../src/data/outlet_gap.json", "w") as fh:
    fh.write(json.dumps({
        "groups": GROUPS,
        "bodyTotals": {o: n for o, n in body_totals.items()},
        "window": {"from": window[0], "to": window[1]},
        "totals": {o: totals[o] for o in totals if totals[o] >= 100},
        "baseline": sorted(SPORTS_BASELINE),
        "people": rows,
    }))

pd.DataFrame([{"player": r["name"], "fox %": round(r["fox"] * 100, 1),
               "sports %": round(r["sports"] * 100, 1), "ratio": r["ratio"]}
              for r in rows])

,player,fox %,sports %,ratio
0,Caitlin Clark,39.7,11.3,3.52
1,Angel Reese,13.5,3.8,3.52
2,Brittney Griner,9.3,4.2,2.22
3,Sophie Cunningham,6.3,0.6,10.20
4,Paige Bueckers,2.5,3.1,0.81
5,DiJonai Carrington,1.4,0.2,6.82
6,A'ja Wilson,1.3,2.5,0.52
7,Natasha Cloud,0.9,0.3,3.16
8,Kelsey Plum,0.9,0.6,1.47
9,Sue Bird,0.9,1.0,0.85


## Spot check

In [11]:
import random

outlet = random.choice(list(rows_by_outlet))

for d, h in random.sample(rows_by_outlet[outlet], min(6, len(rows_by_outlet[outlet]))):
    fired = [t for t, rx in PATTERNS.items() if rx.search(h)]
    print(f"   {d} {h[:78]}")
    print(f"      → {', '.join(fired) or '(none)'}")

   2023-04-04 Jill Biden wants champions LSU, and Iowa, at White House
      → (none)
   2023-08-20 LSU women's star Angel Reese hands out backpacks, school supplies at Baltimore
      → (none)
   2025-12-12 'This is totally different': Sue Bird energized by first camp as Team USA's ma
      → (none)
   2021-05-12 WNBA announces inaugural Commissioner
      → (none)
   2019-08-16 Lynx's Napheesa Collier Having A Stellar Rookie Season
      → (none)
   2025-10-04 WNBA commissioner Cathy Engelbert denies saying Caitlin Clark needs to be 'gra
      → (none)
